# DiffusionBlocks++ on a lab GPU: train once, compare Euler and midpoint

Run this notebook in JupyterLab on a lab computer with an NVIDIA GPU. It trains the same `dblock` classifier and compares Euler and midpoint using one checkpoint. All checkpoints, logs, CSV files, and plots stay on the lab computer; no Google account or Google Drive is needed.

Before opening Jupyter, clone this repository on the lab machine and select a **Python 3.12+ environment with matching CUDA-enabled PyTorch and torchvision**. Use the lab's supported GPU environment. If Jupyter runs remotely, its kernel must run on the GPU machine (and inside your GPU allocation if the lab uses a scheduler).

Open the notebook from the repository root. Run the setup and GPU cells, then W&B setup. The starter run uses 5 `num_epochs` × 3 blocks = 15 Lightning trainer epochs. To evaluate an existing checkpoint, skip training and checkpoint copying and set `CHECKPOINT` and `NUM_BLOCKS` in the sweep configuration.

Internet access is needed for dependency installation, initial dataset downloads, and online W&B logging. Choose a persistent lab storage location below if your home or scratch directories have different retention policies.


## Locate the repository and choose local storage

In [ ]:
from pathlib import Path
import os
import sys

REPO_DIR = Path.cwd().resolve()  # Change to the absolute repository path if needed.
if not (REPO_DIR / "src/diffusion_blocks_plus_plus/deploy_vit.py").is_file():
    raise FileNotFoundError(f"Set REPO_DIR to your DiffusionBlocksPlusPlus checkout: {REPO_DIR}")
os.chdir(REPO_DIR)

# Change this to persistent lab storage if desired. Relative paths use REPO_DIR.
OUTPUT_DIR = (REPO_DIR / "lab_outputs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2  # Keep within your lab job's CPU allocation.
os.environ["USE_TF"] = "0"
print("Kernel Python:", sys.executable)
print("Repository:", REPO_DIR)
print("Outputs:", OUTPUT_DIR)


## Install notebook dependencies

Run once in your selected environment. This leaves the lab's PyTorch installation in place and skips `flash-attn`, which the project does not directly use. Restart the kernel after installation if requested, then rerun the repository setup cell. Dataset downloads use Hugging Face's normal cache on the lab machine; no Google sign-in is involved.


In [ ]:
%pip install 'lightning==2.5.0' 'wandb>=0.22.3' 'transformers==4.52.4' 'datasets==3.5.0' 'huggingface-hub[hf-xet]>=0.31.2' 'hf-transfer>=0.1.9' 'scipy>=1.16.2' pandas matplotlib


## Check the GPU

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select the lab GPU environment/kernel and check your GPU allocation and driver.")
print("GPU:", torch.cuda.get_device_name(0))


## Configure Weights & Biases

Online logging matches the Colab notebook. Authenticate with your W&B account when prompted, or set `WANDB_MODE = "offline"` to run without W&B sign-in. Offline run files are saved under the repository's `logs/` directories and can later be uploaded with `wandb sync <offline-run-directory>`. Offline sweeps may have an empty W&B run-ID column.


In [ ]:
WANDB_MODE = "online"  # "online" or "offline"
if WANDB_MODE not in {"online", "offline"}:
    raise ValueError("WANDB_MODE must be online or offline")
os.environ["WANDB_MODE"] = WANDB_MODE
WANDB_ARGS = ["--debug"] if WANDB_MODE == "offline" else []
if WANDB_MODE == "online":
    import wandb
    wandb.login()


## Train the model

Training writes checkpoints and W&B files under `REPO_DIR/logs/`. The next section copies the latest checkpoint into `OUTPUT_DIR/checkpoints/` for convenient reuse. Reduce batch size if GPU memory is insufficient.


In [ ]:
import subprocess

EPOCHS = 5
BATCH_SIZE = 128
NUM_BLOCKS = 3

subprocess.run([
    sys.executable, "-u", "-m", "src.diffusion_blocks_plus_plus.deploy_vit",
    "train", "cifar100", "--model_type", "dblock",
    "--num_blocks", str(NUM_BLOCKS), "--num_epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE), "--num_workers", str(NUM_WORKERS),
    *WANDB_ARGS,
], cwd=REPO_DIR, check=True)


## Save a checkpoint locally

Review the listed checkpoints. The next cell selects the most recently modified `last.ckpt` from a dblock run. Set `SOURCE` explicitly if you want a different checkpoint. The timestamped copy avoids overwriting checkpoints from earlier runs.


In [ ]:
for checkpoint in sorted((REPO_DIR / "logs").rglob("*.ckpt")):
    print(checkpoint)


In [ ]:
from datetime import datetime, timezone
import shutil

candidates = sorted(
    (REPO_DIR / "logs").glob("*dblock*/last.ckpt"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError("No dblock last.ckpt found under logs/")
SOURCE = candidates[-1]
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DESTINATION = OUTPUT_DIR / "checkpoints" / f"dblock-{stamp}.ckpt"
DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE, DESTINATION)
print(f"Copied {SOURCE} to {DESTINATION}")


## Compare Euler and midpoint across inference sizes

Use the **same checkpoint**, training block count, seed, and evaluation batch size for every combination. `NUM_BLOCKS` must match training; it is independent of `INFERENCE_SIZES`.

In this implementation, `--num_inference_steps N` creates **N sigma points**, hence **N − 1 integration updates**, followed by a final classification call. Euler uses N denoiser calls; midpoint uses 2(N − 1) + 1. Midpoint evaluates its second slope halfway through each interval; Euler uses only the starting slope. These are equal-grid comparisons, not equal-compute comparisons. More points need not improve classification accuracy.

The primary comparison is Euler versus midpoint. To also evaluate Heun, add `"heun"` to `METHODS`; it uses the same denoiser-call count as midpoint.

The defaults run 10 evaluations: two methods × five sizes. For the denser sweep used in our experiments, use `list(range(2, 11)) + [25, 100]`. In online mode, each command creates its own W&B run in `dblocks-cifar100`; `WANDB_RUN_GROUP` groups the sweep. The CSV below provides an explicit method/size-to-W&B-run-ID mapping.

Results are written to a timestamped local directory, under `OUTPUT_DIR` and subsequent sweeps do not overwrite them. Model downloads and dataset preparation can make the first evaluation slower; the recorded duration includes process startup and logging, not just inference.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

CHECKPOINT = OUTPUT_DIR / "checkpoints" / "your-checkpoint.ckpt"
# Use the copy above if you trained in this session:
if "DESTINATION" in globals():
    CHECKPOINT = DESTINATION
CHECKPOINT = CHECKPOINT.expanduser().resolve()
NUM_BLOCKS = 3  # Must match the checkpoint's training configuration.
METHODS = ["euler", "midpoint"]  # Optionally add "heun".
INFERENCE_SIZES = [3, 5, 10, 25, 100]
EVAL_BATCH_SIZE = 128
SEED = 42

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Set CHECKPOINT to an existing local checkpoint: {CHECKPOINT}")
if not METHODS or not INFERENCE_SIZES:
    raise ValueError("Choose at least one method and inference size")
if any(method not in {"euler", "heun", "midpoint"} for method in METHODS):
    raise ValueError("Supported methods: euler, heun, midpoint")
if any(size < 2 for size in INFERENCE_SIZES):
    raise ValueError("Use at least two sigma points for an integration comparison")

SWEEP_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
INFERENCE_RESULTS_DIR = (OUTPUT_DIR / "inference_results") / SWEEP_ID
INFERENCE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results: {INFERENCE_RESULTS_DIR}")

### Run the sweep and save metrics

This uses `subprocess` with ordinary Python arguments rather than Colab `!` interpolation. Each evaluation writes a complete `<method>_<size>.log`. The cell extracts the final Lightning `test/acc` and `test/f1` table into `metrics.csv` after **every run**, recording failures as well. Failed runs remain available for inspection and do not prevent the other combinations from running.

The same seed controls initialization across runs; repeat the sweep with other seeds to measure variation. W&B mode and authentication are inherited from the setup cell. While a command runs, open its W&B run or log to inspect progress.

In [ ]:
import csv
import os
import re
import subprocess
import sys
import time

METRICS_PATH = INFERENCE_RESULTS_DIR / "metrics.csv"
columns = ["method", "inference_steps", "denoiser_calls", "num_blocks", "seed",
           "checkpoint", "test/acc", "test/f1", "elapsed_seconds", "status",
           "returncode", "wandb_run_id", "log_file"]
records = []
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")

for method in METHODS:
    for steps in INFERENCE_SIZES:
        log_path = INFERENCE_RESULTS_DIR / f"{method}_{steps}.log"
        command = [
            sys.executable, "-u", "-m", "src.diffusion_blocks_plus_plus.deploy_vit",
            "test", "cifar100",
            "--model_type", "dblock", "--ckpt_path", str(CHECKPOINT),
            "--num_blocks", str(NUM_BLOCKS), "--num_inference_steps", str(steps),
            "--eval_batch_size", str(EVAL_BATCH_SIZE), "--num_workers", str(NUM_WORKERS),
            "--seed", str(SEED), "--ode", method, *WANDB_ARGS,
        ]
        env = dict(os.environ, USE_TF="0", WANDB_RUN_GROUP=f"solver-sweep-{SWEEP_ID}",
                   WANDB_TAGS=f"{method},sigma-points-{steps},seed-{SEED}")
        # The training module currently chooses its own UUID for the W&B run; recover it
        # from the log below instead of relying on WANDB_RUN_ID.
        print(f"Running {method}, {steps} sigma points → {log_path.name}", flush=True)
        start = time.perf_counter()
        with log_path.open("w") as log:
            completed = subprocess.run(command, cwd=REPO_DIR, env=env, stdout=log,
                                       stderr=subprocess.STDOUT, check=False)
        elapsed = time.perf_counter() - start
        output = ansi_escape.sub("", log_path.read_text())
        metrics = {}
        for metric in ["test/acc", "test/f1"]:
            values = re.findall(re.escape(metric) + r"\s*[│|]\s*([0-9.eE+\-]+)", output)
            metrics[metric] = float(values[-1]) if values else None
        run_ids = re.findall(r"https://wandb\.ai/[^\s/]+/[^\s/]+/runs/([a-zA-Z0-9]+)", output)
        status = "ok" if completed.returncode == 0 and metrics["test/acc"] is not None else "failed"
        records.append(dict(method=method, inference_steps=steps,
                            denoiser_calls=steps if method == "euler" else 2 * steps - 1,
                            num_blocks=NUM_BLOCKS, seed=SEED, checkpoint=str(CHECKPOINT),
                            **metrics, elapsed_seconds=round(elapsed, 2), status=status,
                            returncode=completed.returncode,
                            wandb_run_id=run_ids[-1] if run_ids else "",
                            log_file=str(log_path)))
        with METRICS_PATH.open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns)
            writer.writeheader()
            writer.writerows(records)
        print(f"  {status}: test/acc={metrics['test/acc']}, elapsed={elapsed:.1f}s")
        if status != "ok":
            print("\n".join(output.splitlines()[-15:]))

print(f"Saved {len(records)} evaluations to {METRICS_PATH}")

### Inspect and plot the comparison

Accuracy is displayed as a percentage; the CSV stores the original 0–1 values. The horizontal axis counts sigma points, not denoiser calls. Only successful runs are plotted. A single seed is exploratory evidence, not an uncertainty estimate.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

inference_results = pd.read_csv(METRICS_PATH)
display(inference_results)
successful = inference_results.loc[inference_results["status"] == "ok"]
if successful.empty:
    raise RuntimeError(f"No successful evaluations; inspect logs in {INFERENCE_RESULTS_DIR}")
display(successful.pivot(index="inference_steps", columns="method", values="test/acc"))

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
for method, group in successful.groupby("method"):
    group = group.sort_values("inference_steps")
    ax.plot(group["inference_steps"], group["test/acc"], marker="o", label=method.capitalize())
ticks = sorted(successful["inference_steps"].unique())
ax.set_xscale("log")
ax.set_xticks(ticks, labels=[str(value) for value in ticks])
ax.minorticks_off()
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(xlabel="Inference sigma points (log scale)", ylabel="Test accuracy",
       title="DiffusionBlocks++: solver comparison")
ax.grid(alpha=0.25)
ax.legend()
for extension in ["png", "pdf"]:
    fig.savefig(INFERENCE_RESULTS_DIR / f"test_accuracy.{extension}", dpi=200)
plt.show()
print(f"Logs, metrics.csv, and chart saved in {INFERENCE_RESULTS_DIR}")